# Porównawcza wersja sieci napisana w bibliotece torch

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import torch.utils.benchmark as benchmark
from torchvision import datasets, transforms
import tracemalloc

In [2]:
class CNNNetwork(nn.Module):
    def __init__(self, dropout_p=0.4):
        super(CNNNetwork, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=3, stride=1, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=3, stride=1, padding=0)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 5 * 5, 84)  # After two 3x3 convs and two 2x2 pools on 28x28
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(84, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [3]:
lr = 0.01
batch_size = 1
dropout_p = 0.4
bench_samples = 60_000
num_epochs = 3

In [4]:
def train_epoch(model, x, y, optimizer, criterion, batch_size):
    model.train()
    N = x.size(0)
    total_loss = 0.0
    
    for batch_start in range(0, N, batch_size):
        batch_end = min(batch_start + batch_size, N)
        batch_x = x[batch_start:batch_end]
        batch_y = y[batch_start:batch_end]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss

In [5]:
transform = transforms.ToTensor()
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

x_mini = []
y_mini = []
for i in range(bench_samples):
    x, y = train_data[i]
    x_mini.append(x)
    y_mini.append(y)

x_mini = torch.stack(x_mini)
y_mini = torch.tensor(y_mini)

In [6]:
model = CNNNetwork(dropout_p=dropout_p)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

In [7]:
import time

print("\n[x] Training started...")
for epoch in range(1, num_epochs + 1):
    start_time = time.perf_counter()
    L = train_epoch(model, x_mini, y_mini, optimizer, criterion, batch_size)
    elapsed_time = time.perf_counter() - start_time
    print(f"[+] Epoch {epoch} | Loss: {L:.3f} | Time: {elapsed_time:.2f}s")


[x] Training started...
[+] Epoch 1 | Loss: 37600.394 | Time: 106.59s
[+] Epoch 2 | Loss: 27091.461 | Time: 107.15s
[+] Epoch 3 | Loss: 25027.386 | Time: 109.10s
